# Best Hyperparameters Lookup

Query a training results database (synthetic **or** tiny-imagenet) and return the best `learning_rate` / `dropout` per unit type, ranked by mean validation loss.

In [ ]:
import sqlite3

import pandas as pd

In [ ]:
# Choose the training database to query
# Example folders: synthetic/results and tiny-imagenet/results
db_path = "tiny-imagenet/results/evaluation_26-06-10.db"

In [ ]:
con = sqlite3.connect(db_path)

# Detect the dataset from available columns
cur = con.cursor()
cur.execute("PRAGMA table_info(run_configs)")
columns = {row[1] for row in cur.fetchall()}

is_synthetic = "data_mode" in columns

if is_synthetic:
    print("Detected schema: synthetic")
    group_cols = "c.unit_type, c.data_mode, c.data_type, c.data_dimension"
    select_extra = "c.data_mode, c.data_type, c.data_dimension,"
else:
    print("Detected schema: tiny-imagenet")
    group_cols = "c.unit_type"
    select_extra = ""

query = f"""
    SELECT
        c.unit_type,
        {select_extra}
        c.learning_rate,
        c.dropout,
        c.unit_params,
        AVG(r.best_loss) AS avg_val_loss,
        COUNT(r.id)      AS n_runs
    FROM run_configs c
    JOIN run_results r ON c.id = r.run_id
    GROUP BY {group_cols}, c.learning_rate, c.dropout, c.unit_params
    ORDER BY {group_cols}, avg_val_loss ASC
"""

df_all = pd.read_sql_query(query, con)
con.close()
df_all.head()

In [ ]:
# Keep the lowest-loss config for each group
if is_synthetic:
    group_keys = ["unit_type", "data_mode", "data_type", "data_dimension"]
else:
    group_keys = ["unit_type"]

best = (
    df_all
    .sort_values("avg_val_loss")
    .groupby(group_keys, sort=False)
    .first()
    .reset_index()
    [group_keys + ["learning_rate", "dropout", "unit_params", "avg_val_loss", "n_runs"]]
)

pd.set_option("display.float_format", "{:.6f}".format)
best

In [ ]:
# Print records for pipeline scripts
best_list = best.to_dict(orient="records")
for entry in best_list:
    print(entry)